# Notebook 3: EfficientNetB0 — PyTorch + torchvision
**Brain Tumor MRI Classification — PyTorch Version**

Covers:
- Custom Dataset class with augmentation
- EfficientNetB0 from torchvision (pretrained ImageNet)
- Phase 1: frozen backbone
- Phase 2: fine-tuning top layers
- Training loop with tqdm progress bars
- Full evaluation on test set

## 0. Setup & GPU Check

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import time
import copy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights

import cv2
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

# ── GPU ───────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ── Config ────────────────────────────────────────────────────────
TRAIN_DIR = Path('data/Training')
TEST_DIR  = Path('data/Testing')
CLASSES    = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_EPOCHS_P1 = 15
NUM_EPOCHS_P2 = 20
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)
print('Setup complete.')

PyTorch : 2.6.0+cu124
Device  : cuda
GPU     : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM    : 6.4 GB
Setup complete.


## 1. Dataset Class

In [2]:
from sklearn.model_selection import train_test_split


class BrainTumorDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples   = []
        self.transform = transform
        self.class_to_idx = {cls: i for i, cls in enumerate(classes)}

        for cls in classes:
            cls_dir = Path(base_dir) / cls
            for ext in ['*.jpg', '*.jpeg', '*.png']:
                for p in cls_dir.glob(ext):
                    self.samples.append((p, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Transforms ────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  # ImageNet stats
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Build two underlying datasets so train/val each get the right transform
# (otherwise mutating Subset.dataset.transform also affects the train split)
train_base = BrainTumorDataset(TRAIN_DIR, CLASSES, transform=train_transform)
val_base   = BrainTumorDataset(TRAIN_DIR, CLASSES, transform=val_transform)
test_ds    = BrainTumorDataset(TEST_DIR,  CLASSES, transform=val_transform)

# Stratified 85/15 train/val split — preserves class proportions in both splits
all_indices = np.arange(len(train_base))
all_labels  = np.array([train_base.samples[i][1] for i in all_indices])
train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.15,
    stratify=all_labels,
    random_state=SEED
)

train_ds = torch.utils.data.Subset(train_base, train_indices.tolist())
val_ds   = torch.utils.data.Subset(val_base,   val_indices.tolist())

# Quick sanity check that stratification worked
from collections import Counter
val_dist = Counter([train_base.samples[i][1] for i in val_indices])
print('Val class distribution:', {CLASSES[k]: v for k, v in sorted(val_dist.items())})

# Windows + Jupyter: num_workers=0 avoids spawn issues (set higher only if you know it works)
NUM_WORKERS = 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

Val class distribution: {'glioma': 124, 'meningioma': 124, 'notumor': 59, 'pituitary': 124}
Train: 2439 | Val: 431 | Test: 394


## 2. Class Weights

In [3]:
all_labels = [train_base.samples[i][1] for i in range(len(train_base))]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(CLASSES)),
    y=all_labels
)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print('Class weights:', {CLASSES[i]: round(w, 3) for i, w in enumerate(class_weights)})

Class weights: {'glioma': np.float64(0.869), 'meningioma': np.float64(0.873), 'notumor': np.float64(1.816), 'pituitary': np.float64(0.868)}


## 3. Build Model

In [4]:
def build_efficientnet(num_classes=4, freeze_backbone=True):
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace classifier head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes)
    )
    return model


model = build_efficientnet(freeze_backbone=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainable:,}  (Phase 1 — head only)')

Total params     : 4,336,512
Trainable params : 328,964  (Phase 1 — head only)


## 4. Training Utilities

In [5]:
USE_AMP = torch.cuda.is_available()  # mixed precision when on GPU


def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += imgs.size(0)
    return running_loss / total, correct / total


def run_training(model, train_loader, val_loader, epochs,
                 lr, save_path, history=None,
                 early_stop_patience=6, label_smoothing=0.05):
    """
    Trains with:
      - class-weighted CE loss (+ optional label smoothing)
      - Adam + ReduceLROnPlateau on val_acc
      - mixed-precision (AMP) when CUDA is available
      - early stopping on val_acc (patience = early_stop_patience)
      - best-checkpoint saving by val_acc
    Returns the history dict (extended in place if passed in).
    """
    criterion = nn.CrossEntropyLoss(
        weight=class_weights_tensor,
        label_smoothing=label_smoothing
    )
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    if history is None:
        history = {'train_loss': [], 'val_loss': [],
                   'train_acc':  [], 'val_acc':  []}

    best_val_acc  = 0.0
    best_val_loss = float('inf')
    best_epoch    = 0
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
        val_loss,   val_acc   = eval_epoch( model, val_loader,   criterion,            DEVICE)
        scheduler.step(val_acc)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc  = val_acc
            best_val_loss = val_loss
            best_epoch    = epoch
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            epochs_no_improve += 1

        flag = ' *' if improved else ''
        print(f'Epoch {epoch:02d}/{epochs} | '
              f'Loss {train_loss:.4f}/{val_loss:.4f} | '
              f'Acc {train_acc:.4f}/{val_acc:.4f} | '
              f'lr {optimizer.param_groups[0]["lr"]:.2e} | '
              f'{time.time()-t0:.0f}s{flag}')

        if epochs_no_improve >= early_stop_patience:
            print(f'Early stopping: no val_acc improvement for {early_stop_patience} epochs.')
            break

    print(f'Best val_acc: {best_val_acc:.4f} (val_loss {best_val_loss:.4f}) '
          f'at epoch {best_epoch} — saved to {save_path}')
    return history


print(f'Training utilities ready. AMP: {USE_AMP}')

Training utilities ready. AMP: True


## 5. Phase 1 — Frozen Backbone (head only)

In [ ]:
print('=== PHASE 1: Training head only (backbone frozen) ===')
history = run_training(
    model, train_loader, val_loader,
    epochs=NUM_EPOCHS_P1,
    lr=1e-3,
    save_path='models/efficientnet_phase1.pth'
)

=== PHASE 1: Training head only (backbone frozen) ===


  0%|          | 0/77 [00:00<?, ?it/s]

## 6. Phase 2 — Fine-Tune Top Layers

In [ ]:
import sys
from pathlib import Path
if 'src' not in sys.modules:
    _root = Path.cwd()
    if not (_root / 'src').exists() and (_root.parent / 'src').exists():
        _root = _root.parent
    sys.path.insert(0, str(_root))
from src.models import unfreeze_top, count_trainable

# Load best phase 1 weights
model.load_state_dict(torch.load('models/efficientnet_phase1.pth', map_location=DEVICE))

# Use the modular fine-tuning helper instead of a hand-rolled unfreeze loop.
# unfreeze_top() freezes the whole backbone, then unfreezes the top `frac` of
# backbone params + keeps the classifier head trainable - same intent as the old
# manual loop, but identical to what src/ and train.py do (single source of truth).
model.arch_name = 'efficientnet_b0'      # tag so the helper can locate the head
unfreeze_top(model, frac=0.30)
print(f'Phase 2 trainable params: {count_trainable(model):,}')

print('\n=== PHASE 2: Fine-tuning top layers (lr=1e-5) ===')
history = run_training(
    model, train_loader, val_loader,
    epochs=NUM_EPOCHS_P2,
    lr=1e-5,
    save_path='models/efficientnet_best.pth',
    history=history
)

## 7. Training Curves

In [ ]:
phase1_end = NUM_EPOCHS_P1
epochs_all = range(1, len(history['train_acc']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, train_key, val_key, title, ylim in zip(
        axes,
        ['train_acc', 'train_loss'],
        ['val_acc',   'val_loss'],
        ['Accuracy',  'Loss'],
        [(0.4, 1.02), (0.0, 2.0)]):

    ax.plot(epochs_all, history[train_key], label='Train', lw=2, color='#4361EE')
    ax.plot(epochs_all, history[val_key],   label='Val',   lw=2, color='#F72585')
    ax.axvline(phase1_end, color='gray', lw=1.2, linestyle='--', label='Fine-tune start')
    ax.set_title(f'{title} over epochs', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.legend(); ax.set_ylim(ylim)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('EfficientNetB0 — Phase 1 + Phase 2 training curves', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('results/training_curves.png', bbox_inches='tight')
plt.show()

## 8. Test Set Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('models/efficientnet_best.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Evaluating'):
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        probs   = torch.softmax(outputs, dim=1).cpu().numpy()
        preds   = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print('='*55)
print('EFFICIENTNET TEST RESULTS')
print('='*55)
print(classification_report(all_labels, all_preds, target_names=CLASSES))

auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
print(f'Macro ROC-AUC: {auc:.4f}')

## 9. Confusion Matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, fmt in zip(
        axes, [cm, cm_norm], ['Counts', 'Normalized'], ['d', '.2f']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.5, ax=ax, cbar=False)
    ax.set_title(f'EfficientNetB0 — {title}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('results/efficientnet_confusion_matrix.png', bbox_inches='tight')
plt.show()